# Natural Dataset Individual Shape Analysis

This notebook analyzes the internal landmark shape space of the natural evaluation datasets **BabyLand72** and **InfantFace**.

Important methodological note:
- This notebook **does not** load, fit, or use the synthetic PCA prior.
- PCA is fitted **independently inside each natural dataset** for exploratory dataset characterization only.
- These PCA models are **not** intended for model training.


## How to run

1. Set the dataset paths in the configuration cell.
2. Run the notebook from top to bottom.
3. All tables and figures will be saved under `natural_individual_shape_analysis_outputs/`.
4. The notebook keeps BabyLand72-specific 72-landmark analyses separate from 68-landmark cross-dataset comparisons.


## Configuration

This section defines dataset roots, output paths, plotting style, orientation names, anatomical-group colors, and PCA settings.


In [64]:
from __future__ import annotations

import json
import math
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Iterable

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw

try:
    from scipy import stats
except Exception:
    stats = None

REPO_ROOT = Path("/Users/jocareher/Library/CloudStorage/OneDrive-Personal/Educacion/PhD_UPF_2023/landmarks_detection/")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from scripts.utils.natural_labels import (
    NATURAL_CLASS_ID_TO_ORIENTATION,
    parse_natural_landmark_label,
)
from scripts.utils.visualization import (
    get_landmark_anatomical_group,
    get_landmark_anatomical_label,
    get_landmark_connection_definitions,
)


In [65]:
BABYLAND72_ROOT = Path('/Users/jocareher/Documents/baby_face_72')
INFANTFACE_ROOT = Path('/Users/jocareher/Documents/infanface_adapted/all')
OUTPUT_ROOT = REPO_ROOT / 'natural_individual_shape_analysis_outputs'

ENABLE_BABYLAND72_72_POINT_ANALYSIS = True
COMPARISON_LANDMARK_COUNT = 68
BABYLAND72_LANDMARK_COUNT = 72
INFANTFACE_LANDMARK_COUNT = 68

CLASS_ID_TO_NAME = dict(NATURAL_CLASS_ID_TO_ORIENTATION)
ORIENTATION_ORDER = [CLASS_ID_TO_NAME[index] for index in sorted(CLASS_ID_TO_NAME)]
ORIENTATION_COLORS = {
    'left': '#1f77b4',
    'quarter_left': '#17becf',
    'frontal': '#2ca02c',
    'quarter_right': '#ff7f0e',
    'right': '#d62728',
}
DATASET_COLORS = {
    'BabyLand72': '#355C7D',
    'InfantFace': '#C06C84',
}
ANATOMICAL_GROUP_COLORS = {
    'face_contour': '#4E79A7',
    'right_eyebrow': '#59A14F',
    'left_eyebrow': '#8CD17D',
    'nose_bridge': '#9C755F',
    'nose_base': '#F28E2B',
    'right_eye': '#E15759',
    'left_eye': '#FF9DA7',
    'outer_lip': '#B07AA1',
    'inner_lip': '#D37295',
    'under_lip': '#EDC948',
    'upper_chin': '#76B7B2',
    'left_chin': '#BAB0AC',
    'right_chin': '#86BCB6',
    'unknown': '#999999',
}

FIG_DPI = 220
SAVE_PDF = True
SAVE_TRANSPARENT = False
FONT_SCALE = 1.15

BABYLAND72_PCA_MIN_GLOBAL_SAMPLES = 30
BABYLAND72_PCA_MIN_CLASS_SAMPLES = 10
BABYLAND72_PCA_CANDIDATE_THRESHOLDS = (0.98, 0.95, 0.90, 0.85, 0.80)
BABYLAND72_PCA_MIN_SELECTED_LANDMARKS = 24
PCA_COMPONENTS_TO_PLOT = 6
PAIRWISE_SCORE_COMPONENTS = 5
OUTLIER_TOP_K = 12
PCA_RECONSTRUCTION_VARIANCE_TARGET = 0.95

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
for relative_path in [
    'tables', 'figures', 'outliers',
    'babyland72/tables', 'babyland72/figures', 'babyland72/outliers',
    'infantface/tables', 'infantface/figures', 'infantface/outliers',
    'comparisons/tables', 'comparisons/figures',
]:
    (OUTPUT_ROOT / relative_path).mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    'figure.dpi': FIG_DPI,
    'savefig.dpi': FIG_DPI,
    'font.size': 11 * FONT_SCALE,
    'axes.titlesize': 13 * FONT_SCALE,
    'axes.labelsize': 11 * FONT_SCALE,
    'xtick.labelsize': 10 * FONT_SCALE,
    'ytick.labelsize': 10 * FONT_SCALE,
    'legend.fontsize': 10 * FONT_SCALE,
    'figure.titlesize': 15 * FONT_SCALE,
    'axes.grid': True,
    'grid.alpha': 0.18,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

print('BabyLand72 root:', BABYLAND72_ROOT)
print('InfantFace root:', INFANTFACE_ROOT)
print('Output root:', OUTPUT_ROOT)


BabyLand72 root: /Users/jocareher/Documents/baby_face_72
InfantFace root: /Users/jocareher/Documents/infanface_adapted/all
Output root: /Users/jocareher/Library/CloudStorage/OneDrive-Personal/Educacion/PhD_UPF_2023/landmarks_detection/natural_individual_shape_analysis_outputs


## Label parsers and shared helpers

The notebook uses robust dataset-specific parsers and keeps image-path resolution optional for outlier overlays.


In [66]:
def coordinate_axis_labels(dataset_name: str | None, coordinate_space: str = 'raw') -> tuple[str, str]:
    if coordinate_space == 'normalized':
        return 'normalized x', 'normalized y'
    if dataset_name == 'BabyLand72':
        return 'normalized x', 'normalized y'
    return 'x (pixels)', 'y (pixels)'


def build_connection_segments(landmark_indices: list[int]) -> list[tuple[list[int], bool]]:
    index_to_local = {original_index: local_index for local_index, original_index in enumerate(landmark_indices)}
    segments: list[tuple[list[int], bool]] = []
    for original_range, is_closed in get_landmark_connection_definitions():
        group_indices = [index for index in original_range if index in index_to_local]
        if len(group_indices) < 2:
            continue
        current_segment = [index_to_local[group_indices[0]]]
        for previous_original, current_original in zip(group_indices[:-1], group_indices[1:]):
            if current_original == previous_original + 1:
                current_segment.append(index_to_local[current_original])
            else:
                if len(current_segment) >= 2:
                    segments.append((current_segment, False))
                current_segment = [index_to_local[current_original]]
        if len(current_segment) >= 2:
            close_loop = is_closed and len(group_indices) == len(list(original_range))
            segments.append((current_segment, close_loop))
    return segments


def draw_shape_panel(
    ax: plt.Axes,
    shape: np.ndarray,
    title: str,
    num_landmarks: int,
    dataset_name: str | None = None,
    coordinate_space: str = 'raw',
    valid_landmark_mask: np.ndarray | None = None,
    point_color: str | None = None,
    line_color: str | None = None,
    landmark_indices: list[int] | None = None,
    point_size: float = 20,
    line_width: float = 1.35,
) -> None:
    shape = np.asarray(shape, dtype=np.float64)
    if landmark_indices is None:
        landmark_indices = list(range(num_landmarks))
    else:
        landmark_indices = list(landmark_indices)
    if valid_landmark_mask is None:
        valid_landmark_mask = np.isfinite(shape[:num_landmarks]).all(axis=1)
    else:
        valid_landmark_mask = np.asarray(valid_landmark_mask[:num_landmarks], dtype=bool)

    for local_segment, is_closed in build_connection_segments(landmark_indices[:num_landmarks]):
        segment_points = []
        segment_original_indices = []
        for local_index in local_segment:
            if local_index >= len(valid_landmark_mask) or not valid_landmark_mask[local_index]:
                segment_points = []
                segment_original_indices = []
                break
            point = shape[local_index]
            if not np.isfinite(point).all():
                segment_points = []
                segment_original_indices = []
                break
            segment_points.append(tuple(point.tolist()))
            segment_original_indices.append(landmark_indices[local_index])
        if len(segment_points) < 2:
            continue
        group_name = get_landmark_anatomical_group(segment_original_indices[0])
        current_color = line_color or ANATOMICAL_GROUP_COLORS.get(group_name, ANATOMICAL_GROUP_COLORS['unknown'])
        xs, ys = zip(*segment_points)
        ax.plot(xs, ys, color=current_color, linewidth=line_width, alpha=0.9)
        if is_closed and len(segment_points) >= 3:
            ax.plot(
                [segment_points[-1][0], segment_points[0][0]],
                [segment_points[-1][1], segment_points[0][1]],
                color=current_color,
                linewidth=line_width,
                alpha=0.9,
            )

    for local_index, (x, y) in enumerate(shape[:num_landmarks]):
        if local_index >= len(valid_landmark_mask) or not valid_landmark_mask[local_index]:
            continue
        if not np.isfinite(x) or not np.isfinite(y):
            continue
        original_index = landmark_indices[local_index]
        group = get_landmark_anatomical_group(original_index)
        current_color = point_color or ANATOMICAL_GROUP_COLORS.get(group, ANATOMICAL_GROUP_COLORS['unknown'])
        ax.scatter(x, y, s=point_size, color=current_color, edgecolor='white', linewidths=0.35, zorder=3)

    x_label, y_label = coordinate_axis_labels(dataset_name, coordinate_space)
    ax.set_title(title)
    ax.set_aspect('equal')
    ax.invert_yaxis()
    ax.set_xlabel(x_label)
    ax.set_ylabel(y_label)


def apply_shared_shape_limits(axes: Iterable[plt.Axes], shapes: Iterable[np.ndarray], padding_ratio: float = 0.08) -> None:
    finite_points = []
    for shape in shapes:
        current = np.asarray(shape, dtype=np.float64)
        mask = np.isfinite(current).all(axis=1)
        if mask.any():
            finite_points.append(current[mask])
    if not finite_points:
        return
    stacked = np.vstack(finite_points)
    min_xy = stacked.min(axis=0)
    max_xy = stacked.max(axis=0)
    span = np.maximum(max_xy - min_xy, 1e-6)
    padding = span * padding_ratio
    for ax in axes:
        ax.set_xlim(min_xy[0] - padding[0], max_xy[0] + padding[0])
        ax.set_ylim(max_xy[1] + padding[1], min_xy[1] - padding[1])


def plot_shape(
    ax: plt.Axes,
    shape: np.ndarray,
    title: str,
    num_landmarks: int,
    dataset_name: str | None = None,
    coordinate_space: str = 'raw',
    valid_landmark_mask: np.ndarray | None = None,
    landmark_indices: list[int] | None = None,
) -> None:
    draw_shape_panel(
        ax=ax,
        shape=shape,
        title=title,
        num_landmarks=num_landmarks,
        dataset_name=dataset_name,
        coordinate_space=coordinate_space,
        valid_landmark_mask=valid_landmark_mask,
        landmark_indices=landmark_indices,
    )


def is_fitted_pca_result(pca_result: dict[str, Any] | None) -> bool:
    if pca_result is None or not isinstance(pca_result, dict):
        return False
    required_keys = {'scores', 'explained_variance', 'components', 'mean_vector', 'prepared_shapes'}
    return required_keys.issubset(pca_result.keys())


## Load datasets

This section parses all labels, reports malformed files, and prepares 68-landmark trimmed views for direct comparison.


In [67]:
babyland72_samples, babyland72_errors = load_samples(BABYLAND72_ROOT, 'BabyLand72')
infantface_samples, infantface_errors = load_samples(INFANTFACE_ROOT, 'InfantFace')

babyland72_samples_68 = [trim_sample(sample, COMPARISON_LANDMARK_COUNT) for sample in babyland72_samples]
infantface_samples_68 = [trim_sample(sample, COMPARISON_LANDMARK_COUNT) for sample in infantface_samples]

print('BabyLand72 samples:', len(babyland72_samples), '| parse errors:', len(babyland72_errors))
print('InfantFace samples:', len(infantface_samples), '| parse errors:', len(infantface_errors))


BabyLand72 samples: 311 | parse errors: 0
InfantFace samples: 405 | parse errors: 0


## Dataset sanity checks

These checks summarize sample counts, coordinate ranges, bounding-box statistics, malformed labels, and suspicious samples.


In [68]:
def build_sample_level_table(samples: list[NaturalDatasetSample], num_landmarks: int) -> pd.DataFrame:
    rows = []
    for sample in samples:
        mask = valid_mask(sample, num_landmarks=num_landmarks)
        bbox_stats = compute_bbox_stats(sample.landmarks[:num_landmarks], mask)
        coords = sample.landmarks[:num_landmarks]
        finite_mask = np.isfinite(coords).all(axis=1)
        duplicate_rows = 0
        if finite_mask.any():
            unique_rows = np.unique(coords[finite_mask], axis=0)
            duplicate_rows = int(finite_mask.sum() - len(unique_rows))
        row = {
            'dataset': sample.dataset_name,
            'image_id': sample.image_id,
            'class_idx': sample.class_idx,
            'orientation': sample.orientation,
            'num_landmarks': num_landmarks,
            'valid_landmark_count': int(mask.sum()),
            'finite_landmark_count': int(finite_mask.sum()),
            'duplicate_coordinate_rows': duplicate_rows,
            'min_x': float(np.nanmin(coords[:, 0])),
            'max_x': float(np.nanmax(coords[:, 0])),
            'min_y': float(np.nanmin(coords[:, 1])),
            'max_y': float(np.nanmax(coords[:, 1])),
            'has_degenerate_bbox': bool(mask.sum() >= 2 and (bbox_stats['bbox_width'] <= 1e-8 or bbox_stats['bbox_height'] <= 1e-8)),
            'has_extreme_coordinates': bool(np.nanmax(np.abs(coords)) > 5000),
            'too_few_valid_landmarks': bool(mask.sum() < max(10, int(0.5 * num_landmarks))),
        }
        row.update(bbox_stats)
        rows.append(row)
    return pd.DataFrame(rows)


def build_dataset_summary(samples: list[NaturalDatasetSample], sample_level_df: pd.DataFrame) -> pd.DataFrame:
    orientation_counts = sample_level_df['orientation'].value_counts().reindex(ORIENTATION_ORDER, fill_value=0)
    rows = [{
        'dataset': samples[0].dataset_name if samples else 'unknown',
        'num_samples': len(samples),
        'num_orientations_present': int((orientation_counts > 0).sum()),
        'mean_valid_landmark_count': float(sample_level_df['valid_landmark_count'].mean()) if not sample_level_df.empty else np.nan,
        'median_valid_landmark_count': float(sample_level_df['valid_landmark_count'].median()) if not sample_level_df.empty else np.nan,
        'mean_bbox_width': float(sample_level_df['bbox_width'].mean()) if 'bbox_width' in sample_level_df else np.nan,
        'mean_bbox_height': float(sample_level_df['bbox_height'].mean()) if 'bbox_height' in sample_level_df else np.nan,
        'mean_bbox_diagonal': float(sample_level_df['bbox_diagonal'].mean()) if 'bbox_diagonal' in sample_level_df else np.nan,
        'mean_bbox_area': float(sample_level_df['bbox_area'].mean()) if 'bbox_area' in sample_level_df else np.nan,
        'degenerate_bbox_samples': int(sample_level_df['has_degenerate_bbox'].sum()) if 'has_degenerate_bbox' in sample_level_df else 0,
        'extreme_coordinate_samples': int(sample_level_df['has_extreme_coordinates'].sum()) if 'has_extreme_coordinates' in sample_level_df else 0,
        'too_few_valid_landmarks_samples': int(sample_level_df['too_few_valid_landmarks'].sum()) if 'too_few_valid_landmarks' in sample_level_df else 0,
    }]
    for orientation, count in orientation_counts.items():
        rows[0][f'count_{orientation}'] = int(count)
        rows[0][f'percent_{orientation}'] = float(100.0 * count / max(len(samples), 1))
    return pd.DataFrame(rows)


def build_visibility_summary(samples: list[NaturalDatasetSample], num_landmarks: int) -> pd.DataFrame:
    rows = []
    if not samples or samples[0].visibility is None:
        return pd.DataFrame(rows)
    for landmark_idx in range(num_landmarks):
        for orientation in ORIENTATION_ORDER + ['all']:
            subset = samples if orientation == 'all' else [sample for sample in samples if sample.orientation == orientation]
            if not subset:
                continue
            visibility = np.asarray([sample.visibility[landmark_idx] for sample in subset], dtype=np.float64)
            rows.append({
                'landmark_idx': landmark_idx,
                'anatomical_group': get_landmark_anatomical_group(landmark_idx),
                'anatomical_label': get_landmark_anatomical_label(landmark_idx),
                'orientation': orientation,
                'visibility_rate': float(visibility.mean()),
                'visible_count': int(visibility.sum()),
                'sample_count': int(len(subset)),
            })
    return pd.DataFrame(rows)


def build_parse_error_table(errors: list[dict[str, Any]]) -> pd.DataFrame:
    return pd.DataFrame(errors)


babyland72_sample_summary = build_sample_level_table(babyland72_samples, BABYLAND72_LANDMARK_COUNT)
infantface_sample_summary = build_sample_level_table(infantface_samples, INFANTFACE_LANDMARK_COUNT)

babyland72_dataset_summary = build_dataset_summary(babyland72_samples, babyland72_sample_summary)
infantface_dataset_summary = build_dataset_summary(infantface_samples, infantface_sample_summary)

babyland72_visibility_summary = build_visibility_summary(babyland72_samples, BABYLAND72_LANDMARK_COUNT)
parse_error_table = pd.concat(
    [build_parse_error_table(babyland72_errors), build_parse_error_table(infantface_errors)],
    ignore_index=True,
) if (babyland72_errors or infantface_errors) else pd.DataFrame(columns=['dataset', 'label_path', 'error'])

save_table(babyland72_dataset_summary, OUTPUT_ROOT / 'babyland72' / 'tables' / 'babyland72_dataset_summary.csv')
save_table(infantface_dataset_summary, OUTPUT_ROOT / 'infantface' / 'tables' / 'infantface_dataset_summary.csv')
save_table(babyland72_sample_summary, OUTPUT_ROOT / 'babyland72' / 'tables' / 'babyland72_coordinate_sanity.csv')
save_table(infantface_sample_summary, OUTPUT_ROOT / 'infantface' / 'tables' / 'infantface_coordinate_sanity.csv')
save_table(babyland72_visibility_summary, OUTPUT_ROOT / 'babyland72' / 'tables' / 'babyland72_visibility_summary.csv')
save_table(parse_error_table, OUTPUT_ROOT / 'tables' / 'label_parse_errors.csv')

babyland72_dataset_summary


,dataset,num_samples,num_orientations_present,mean_valid_landmark_count,median_valid_landmark_count,mean_bbox_width,mean_bbox_height,mean_bbox_diagonal,mean_bbox_area,degenerate_bbox_samples,...,count_left,percent_left,count_quarter_left,percent_quarter_left,count_frontal,percent_frontal,count_quarter_right,percent_quarter_right,count_right,percent_right
0,BabyLand72,311,5,48.093248,48.0,0.301764,0.35853,0.476533,0.112247,0,...,81,26.045016,31,9.967846,83,26.688103,23,7.395498,93,29.903537


## Orientation distribution

These plots summarize orientation balance inside each dataset and compare BabyLand72 versus InfantFace.


In [69]:
def orientation_count_table(samples: list[NaturalDatasetSample], dataset_name: str) -> pd.DataFrame:
    counts = pd.Series([sample.orientation for sample in samples], dtype='object').value_counts().reindex(ORIENTATION_ORDER, fill_value=0)
    table = pd.DataFrame({
        'dataset': dataset_name,
        'orientation': counts.index,
        'count': counts.values,
    })
    table['percentage'] = 100.0 * table['count'] / max(table['count'].sum(), 1)
    return table


def plot_orientation_bars(table: pd.DataFrame, dataset_name: str, output_stem: Path) -> None:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), constrained_layout=True)
    colors = [ORIENTATION_COLORS[label] for label in table['orientation']]
    axes[0].bar(table['orientation'], table['count'], color=colors)
    axes[0].set_title(f'{dataset_name}: sample counts by orientation')
    axes[0].set_ylabel('Sample count')
    axes[0].tick_params(axis='x', rotation=25)
    axes[1].bar(table['orientation'], table['percentage'], color=colors)
    axes[1].set_title(f'{dataset_name}: percentage by orientation')
    axes[1].set_ylabel('Percentage (%)')
    axes[1].tick_params(axis='x', rotation=25)
    save_figure(fig, output_stem)


def plot_orientation_comparison(baby_table: pd.DataFrame, infant_table: pd.DataFrame) -> None:
    merged = baby_table[['orientation', 'count', 'percentage']].merge(
        infant_table[['orientation', 'count', 'percentage']],
        on='orientation',
        suffixes=('_babyland72', '_infantface'),
    )
    x = np.arange(len(merged))
    width = 0.36
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
    axes[0].bar(x - width / 2, merged['count_babyland72'], width=width, color=DATASET_COLORS['BabyLand72'], label='BabyLand72')
    axes[0].bar(x + width / 2, merged['count_infantface'], width=width, color=DATASET_COLORS['InfantFace'], label='InfantFace')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(merged['orientation'], rotation=25)
    axes[0].set_title('Orientation count comparison')
    axes[0].set_ylabel('Sample count')
    axes[0].legend(frameon=False)

    axes[1].bar(['BabyLand72', 'InfantFace'], [100, 100], color='none')
    cumulative = np.zeros(2)
    for orientation in ORIENTATION_ORDER:
        values = np.array([
            merged.loc[merged['orientation'] == orientation, 'percentage_babyland72'].iloc[0],
            merged.loc[merged['orientation'] == orientation, 'percentage_infantface'].iloc[0],
        ])
        axes[1].bar(['BabyLand72', 'InfantFace'], values, bottom=cumulative, color=ORIENTATION_COLORS[orientation], label=orientation)
        cumulative += values
    axes[1].set_title('Orientation percentage comparison')
    axes[1].set_ylabel('Percentage (%)')
    axes[1].legend(frameon=False, bbox_to_anchor=(1.02, 1.0), loc='upper left')
    save_figure(fig, OUTPUT_ROOT / 'comparisons' / 'figures' / 'orientation_comparison')


baby_orientation_table = orientation_count_table(babyland72_samples, 'BabyLand72')
infant_orientation_table = orientation_count_table(infantface_samples, 'InfantFace')

save_table(baby_orientation_table, OUTPUT_ROOT / 'babyland72' / 'tables' / 'babyland72_orientation_distribution.csv')
save_table(infant_orientation_table, OUTPUT_ROOT / 'infantface' / 'tables' / 'infantface_orientation_distribution.csv')
plot_orientation_bars(baby_orientation_table, 'BabyLand72', OUTPUT_ROOT / 'babyland72' / 'figures' / 'orientation_distribution')
plot_orientation_bars(infant_orientation_table, 'InfantFace', OUTPUT_ROOT / 'infantface' / 'figures' / 'orientation_distribution')
plot_orientation_comparison(baby_orientation_table, infant_orientation_table)

baby_orientation_table


,dataset,orientation,count,percentage
0,BabyLand72,left,81,26.045016
1,BabyLand72,quarter_left,31,9.967846
2,BabyLand72,frontal,83,26.688103
3,BabyLand72,quarter_right,23,7.395498
4,BabyLand72,right,93,29.903537


## Mean shape analysis

Mean-shape plots are computed per dataset and per orientation. BabyLand72 uses only valid visible coordinates for mean-coordinate estimation.


In [70]:
def compute_mean_shape(samples: list[NaturalDatasetSample], num_landmarks: int) -> tuple[np.ndarray, np.ndarray]:
    sums = np.zeros((num_landmarks, 2), dtype=np.float64)
    counts = np.zeros(num_landmarks, dtype=np.int64)
    for sample in samples:
        coords = sample.landmarks[:num_landmarks]
        mask = valid_mask(sample, num_landmarks=num_landmarks)
        sums[mask] += coords[mask]
        counts[mask] += 1
    mean_shape = np.full((num_landmarks, 2), np.nan, dtype=np.float64)
    valid = counts > 0
    mean_shape[valid] = sums[valid] / counts[valid, None]
    return mean_shape, counts


def compute_mean_shape_table(
    samples: list[NaturalDatasetSample],
    num_landmarks: int,
    dataset_name: str,
    class_idx: int | None,
    orientation: str,
) -> pd.DataFrame:
    mean_shape, counts = compute_mean_shape(samples, num_landmarks)
    rows = []
    for landmark_idx in range(num_landmarks):
        visibility_rate = np.nan
        if samples and samples[0].visibility is not None:
            visibility_values = np.asarray([sample.visibility[landmark_idx] for sample in samples], dtype=np.float64)
            visibility_rate = float(visibility_values.mean())
        rows.append({
            'dataset': dataset_name,
            'class_idx': class_idx,
            'orientation': orientation,
            'landmark_idx': landmark_idx,
            'anatomical_group': get_landmark_anatomical_group(landmark_idx),
            'anatomical_label': get_landmark_anatomical_label(landmark_idx),
            'mean_x': float(mean_shape[landmark_idx, 0]) if np.isfinite(mean_shape[landmark_idx, 0]) else np.nan,
            'mean_y': float(mean_shape[landmark_idx, 1]) if np.isfinite(mean_shape[landmark_idx, 1]) else np.nan,
            'valid_count': int(counts[landmark_idx]),
            'visibility_rate': visibility_rate,
        })
    return pd.DataFrame(rows)


def normalize_shape_center_scale_masked(shape: np.ndarray, mask: np.ndarray, eps: float = 1e-8) -> tuple[np.ndarray, np.ndarray]:
    normalized = np.full_like(shape, np.nan, dtype=np.float64)
    if mask.sum() < 2:
        return normalized, mask
    coords = shape[mask]
    centroid = coords.mean(axis=0)
    centered = shape - centroid
    scale = float(np.sqrt(np.mean(np.sum(centered[mask] ** 2, axis=1))))
    scale = max(scale, eps)
    normalized[mask] = centered[mask] / scale
    return normalized, mask.copy()


def estimate_similarity_transform_masked(
    source: np.ndarray,
    target: np.ndarray,
    mask: np.ndarray,
    allow_reflection: bool = False,
    eps: float = 1e-8,
) -> tuple[np.ndarray, float, np.ndarray]:
    source_coords = source[mask]
    target_coords = target[mask]
    if len(source_coords) < 2 or len(target_coords) < 2:
        return np.eye(2), 1.0, np.zeros(2, dtype=np.float64)
    source_mean = source_coords.mean(axis=0)
    target_mean = target_coords.mean(axis=0)
    source_centered = source_coords - source_mean
    target_centered = target_coords - target_mean
    source_norm = math.sqrt(max(float(np.sum(source_centered ** 2)), eps))
    target_norm = math.sqrt(max(float(np.sum(target_centered ** 2)), eps))
    source_normalized = source_centered / source_norm
    target_normalized = target_centered / target_norm
    covariance = source_normalized.T @ target_normalized
    u_matrix, _, vh_matrix = np.linalg.svd(covariance, full_matrices=False)
    rotation = vh_matrix.T @ u_matrix.T
    if (not allow_reflection) and np.linalg.det(rotation) < 0:
        vh_matrix[-1, :] *= -1
        rotation = vh_matrix.T @ u_matrix.T
    scale = target_norm / max(source_norm, eps)
    translation = target_mean - scale * (source_mean @ rotation)
    return rotation, scale, translation


def apply_similarity_transform_masked(
    shape: np.ndarray,
    mask: np.ndarray,
    rotation: np.ndarray,
    scale: float,
    translation: np.ndarray,
) -> np.ndarray:
    transformed = np.full_like(shape, np.nan, dtype=np.float64)
    transformed[mask] = scale * (shape[mask] @ rotation) + translation
    return transformed


def compute_normalized_mean_shape(
    samples: list[NaturalDatasetSample],
    num_landmarks: int,
    method: str,
    landmark_indices: list[int] | None = None,
) -> tuple[np.ndarray, np.ndarray, int]:
    if landmark_indices is None:
        landmark_indices = list(range(num_landmarks))
    shapes = [sample.landmarks[landmark_indices].astype(np.float64, copy=True) for sample in samples]
    masks = [valid_mask(sample, num_landmarks=num_landmarks)[landmark_indices].copy() for sample in samples]
    usable_pairs = [(shape, mask) for shape, mask in zip(shapes, masks) if mask.sum() >= 2]
    if not usable_pairs:
        return np.full((len(landmark_indices), 2), np.nan, dtype=np.float64), np.zeros(len(landmark_indices), dtype=np.int64), 0

    if method == 'center_scale':
        normalized_shapes = []
        for shape, mask in usable_pairs:
            normalized_shape, _ = normalize_shape_center_scale_masked(shape, mask)
            normalized_shapes.append((normalized_shape, mask))
    elif method == 'procrustes':
        reference_shape, _ = normalize_shape_center_scale_masked(usable_pairs[0][0], usable_pairs[0][1])
        current_reference = reference_shape.copy()
        aligned_pairs = []
        for _ in range(50):
            aligned_pairs = []
            for shape, mask in usable_pairs:
                normalized_shape, normalized_mask = normalize_shape_center_scale_masked(shape, mask)
                overlap = normalized_mask & np.isfinite(current_reference).all(axis=1)
                if overlap.sum() < 2:
                    continue
                rotation, scale, translation = estimate_similarity_transform_masked(
                    normalized_shape,
                    current_reference,
                    overlap,
                    allow_reflection=False,
                )
                aligned_shape = apply_similarity_transform_masked(
                    normalized_shape,
                    normalized_mask,
                    rotation,
                    scale,
                    translation,
                )
                aligned_pairs.append((aligned_shape, normalized_mask))
            if not aligned_pairs:
                break
            sums = np.zeros((len(landmark_indices), 2), dtype=np.float64)
            counts = np.zeros(len(landmark_indices), dtype=np.int64)
            for aligned_shape, aligned_mask in aligned_pairs:
                sums[aligned_mask] += aligned_shape[aligned_mask]
                counts[aligned_mask] += 1
            updated_reference = np.full((len(landmark_indices), 2), np.nan, dtype=np.float64)
            valid_ref = counts > 0
            updated_reference[valid_ref] = sums[valid_ref] / counts[valid_ref, None]
            updated_reference, _ = normalize_shape_center_scale_masked(updated_reference, valid_ref)
            difference = updated_reference - current_reference
            if np.nanmax(np.abs(np.nan_to_num(difference, nan=0.0))) < 1e-6:
                current_reference = updated_reference
                break
            current_reference = updated_reference
        normalized_shapes = aligned_pairs
    else:
        raise ValueError(f'Unknown normalization method: {method}')

    sums = np.zeros((len(landmark_indices), 2), dtype=np.float64)
    counts = np.zeros(len(landmark_indices), dtype=np.int64)
    for normalized_shape, mask in normalized_shapes:
        valid_current = mask & np.isfinite(normalized_shape).all(axis=1)
        sums[valid_current] += normalized_shape[valid_current]
        counts[valid_current] += 1
    mean_shape = np.full((len(landmark_indices), 2), np.nan, dtype=np.float64)
    valid = counts > 0
    mean_shape[valid] = sums[valid] / counts[valid, None]
    return mean_shape, counts, len(normalized_shapes)


def plot_orientation_mean_shapes_visible_only(
    samples: list[NaturalDatasetSample],
    dataset_name: str,
    num_landmarks: int,
    output_stem: Path,
) -> pd.DataFrame:
    fig, axes = plt.subplots(2, 3, figsize=(15, 9), constrained_layout=True)
    orientation_tables = []
    plotted_shapes = []
    global_table = compute_mean_shape_table(samples, num_landmarks, dataset_name, None, 'all_orientations_mixed')
    global_mean, global_counts = compute_mean_shape(samples, num_landmarks)
    plotted_shapes.append(global_mean)
    plot_shape(
        axes.flat[0],
        global_mean,
        f'{dataset_name}: global visible-only mean\n(mixes orientations)',
        num_landmarks,
        dataset_name=dataset_name,
        coordinate_space='raw',
        valid_landmark_mask=global_counts > 0,
    )
    axes.flat[0].text(
        0.02,
        0.02,
        f'min/max valid = {int(np.nanmin(global_counts))}/{int(np.nanmax(global_counts))}',
        transform=axes.flat[0].transAxes,
        fontsize=10,
    )
    orientation_tables.append(global_table)
    for axis, (class_idx, orientation) in zip(axes.flat[1:], CLASS_ID_TO_NAME.items()):
        subset = [sample for sample in samples if sample.class_idx == class_idx]
        mean_shape, counts = compute_mean_shape(subset, num_landmarks)
        plotted_shapes.append(mean_shape)
        plot_shape(
            axis,
            mean_shape,
            f'{orientation} | visible-only mean',
            num_landmarks,
            dataset_name=dataset_name,
            coordinate_space='raw',
            valid_landmark_mask=counts > 0,
        )
        axis.text(0.02, 0.02, f'n={len(subset)}\nmin valid={int(np.nanmin(counts)) if len(counts) else 0}', transform=axis.transAxes, fontsize=10)
        orientation_tables.append(compute_mean_shape_table(subset, num_landmarks, dataset_name, class_idx, orientation))
    apply_shared_shape_limits(axes.flat, plotted_shapes)
    save_figure(fig, output_stem)
    return pd.concat(orientation_tables, ignore_index=True)


def plot_global_mean_shape(shape: np.ndarray, valid_counts: np.ndarray, title: str, output_stem: Path, dataset_name: str, coordinate_space: str, landmark_indices: list[int] | None = None) -> None:
    fig, ax = plt.subplots(figsize=(6.3, 6.3), constrained_layout=True)
    plot_shape(
        ax,
        shape,
        title,
        len(shape),
        dataset_name=dataset_name,
        coordinate_space=coordinate_space,
        valid_landmark_mask=valid_counts > 0,
        landmark_indices=landmark_indices,
    )
    apply_shared_shape_limits([ax], [shape])
    ax.text(0.02, 0.02, f'min/max valid = {int(np.nanmin(valid_counts))}/{int(np.nanmax(valid_counts))}', transform=ax.transAxes, fontsize=10)
    save_figure(fig, output_stem)


def plot_overlay_normalized_mean_shapes(
    shapes: dict[str, tuple[np.ndarray, np.ndarray]],
    title: str,
    output_stem: Path,
    landmark_indices: list[int] | None = None,
) -> None:
    fig, ax = plt.subplots(figsize=(7.2, 7.2), constrained_layout=True)
    plotted_shapes = []
    if landmark_indices is None:
        first_shape = next(iter(shapes.values()))[0]
        landmark_indices = list(range(len(first_shape)))
    for dataset_name, (shape, counts) in shapes.items():
        valid = counts > 0
        draw_shape_panel(
            ax=ax,
            shape=shape,
            title=title,
            num_landmarks=len(shape),
            dataset_name=dataset_name,
            coordinate_space='normalized',
            valid_landmark_mask=valid,
            point_color=DATASET_COLORS[dataset_name],
            line_color=DATASET_COLORS[dataset_name],
            landmark_indices=landmark_indices,
            point_size=18,
            line_width=1.5,
        )
        plotted_shapes.append(shape)
    apply_shared_shape_limits([ax], plotted_shapes)
    ax.legend(
        handles=[
            plt.Line2D([0], [0], color=DATASET_COLORS['BabyLand72'], label='BabyLand72'),
            plt.Line2D([0], [0], color=DATASET_COLORS['InfantFace'], label='InfantFace'),
        ],
        frameon=False,
        bbox_to_anchor=(1.02, 1.0),
        loc='upper left',
    )
    save_figure(fig, output_stem)


babyland72_mean_shape_by_orientation = plot_orientation_mean_shapes_visible_only(
    babyland72_samples,
    'BabyLand72',
    BABYLAND72_LANDMARK_COUNT,
    OUTPUT_ROOT / 'babyland72' / 'figures' / 'mean_shape_by_orientation_visible_only',
)
babyland72_mean_shape_by_orientation_68 = plot_orientation_mean_shapes_visible_only(
    babyland72_samples_68,
    'BabyLand72',
    COMPARISON_LANDMARK_COUNT,
    OUTPUT_ROOT / 'babyland72' / 'figures' / 'mean_shape_by_orientation_visible_only_68',
)
infantface_mean_shape_by_orientation = plot_orientation_mean_shapes_visible_only(
    infantface_samples_68,
    'InfantFace',
    COMPARISON_LANDMARK_COUNT,
    OUTPUT_ROOT / 'infantface' / 'figures' / 'mean_shape_by_orientation',
)

save_table(
    babyland72_mean_shape_by_orientation,
    OUTPUT_ROOT / 'babyland72' / 'tables' / 'mean_shape_by_orientation_visible_only.csv',
)
save_table(
    babyland72_mean_shape_by_orientation_68,
    OUTPUT_ROOT / 'babyland72' / 'tables' / 'mean_shape_by_orientation_visible_only_68.csv',
)
save_table(
    infantface_mean_shape_by_orientation,
    OUTPUT_ROOT / 'infantface' / 'tables' / 'mean_shape_by_orientation.csv',
)

baby_global_visible_only_shape, baby_global_visible_only_counts = compute_mean_shape(babyland72_samples, BABYLAND72_LANDMARK_COUNT)
plot_global_mean_shape(
    baby_global_visible_only_shape,
    baby_global_visible_only_counts,
    'BabyLand72: global visible-only mean (raw coordinates)',
    OUTPUT_ROOT / 'babyland72' / 'figures' / 'mean_shape_global_visible_only',
    dataset_name='BabyLand72',
    coordinate_space='raw',
)

stable_common_landmarks = pca_experiments['BabyLand72']['subset_68']['selected_landmarks']
stable_landmark_summary = pd.DataFrame({
    'landmark_idx': stable_common_landmarks,
    'anatomical_group': [get_landmark_anatomical_group(index) for index in stable_common_landmarks],
    'anatomical_label': [get_landmark_anatomical_label(index) for index in stable_common_landmarks],
})
save_table(
    stable_landmark_summary,
    OUTPUT_ROOT / 'comparisons' / 'tables' / 'stable_common_landmarks_68.csv',
)

baby_center_scale_mean_68, baby_center_scale_counts_68, baby_center_scale_sample_count_68 = compute_normalized_mean_shape(
    babyland72_samples_68,
    COMPARISON_LANDMARK_COUNT,
    method='center_scale',
    landmark_indices=stable_common_landmarks,
)
baby_procrustes_mean_68, baby_procrustes_counts_68, baby_procrustes_sample_count_68 = compute_normalized_mean_shape(
    babyland72_samples_68,
    COMPARISON_LANDMARK_COUNT,
    method='procrustes',
    landmark_indices=stable_common_landmarks,
)
infant_center_scale_mean_68, infant_center_scale_counts_68, infant_center_scale_sample_count_68 = compute_normalized_mean_shape(
    infantface_samples_68,
    COMPARISON_LANDMARK_COUNT,
    method='center_scale',
    landmark_indices=stable_common_landmarks,
)
infant_procrustes_mean_68, infant_procrustes_counts_68, infant_procrustes_sample_count_68 = compute_normalized_mean_shape(
    infantface_samples_68,
    COMPARISON_LANDMARK_COUNT,
    method='procrustes',
    landmark_indices=stable_common_landmarks,
)

plot_global_mean_shape(
    baby_center_scale_mean_68,
    baby_center_scale_counts_68,
    'BabyLand72: stable-subset center-scale mean\n(68-landmark space, visibility-stable subset)',
    OUTPUT_ROOT / 'babyland72' / 'figures' / 'mean_shape_global_center_scale',
    dataset_name='BabyLand72',
    coordinate_space='normalized',
    landmark_indices=stable_common_landmarks,
)
plot_global_mean_shape(
    baby_procrustes_mean_68,
    baby_procrustes_counts_68,
    'BabyLand72: stable-subset Procrustes mean\n(68-landmark space, visibility-stable subset)',
    OUTPUT_ROOT / 'babyland72' / 'figures' / 'mean_shape_global_procrustes',
    dataset_name='BabyLand72',
    coordinate_space='normalized',
    landmark_indices=stable_common_landmarks,
)
plot_overlay_normalized_mean_shapes(
    {
        'BabyLand72': (baby_center_scale_mean_68, baby_center_scale_counts_68),
        'InfantFace': (infant_center_scale_mean_68, infant_center_scale_counts_68),
    },
    'BabyLand72 vs InfantFace | center-scale normalized\n(stable BabyLand72 landmark subset)',
    OUTPUT_ROOT / 'comparisons' / 'figures' / 'babyland72_vs_infantface_mean_shape_common_space_center_scale',
    landmark_indices=stable_common_landmarks,
)
plot_overlay_normalized_mean_shapes(
    {
        'BabyLand72': (baby_procrustes_mean_68, baby_procrustes_counts_68),
        'InfantFace': (infant_procrustes_mean_68, infant_procrustes_counts_68),
    },
    'BabyLand72 vs InfantFace | Procrustes normalized\n(stable BabyLand72 landmark subset)',
    OUTPUT_ROOT / 'comparisons' / 'figures' / 'babyland72_vs_infantface_mean_shape_common_space_procrustes',
    landmark_indices=stable_common_landmarks,
)

babyland72_validity_summary = babyland72_mean_shape_by_orientation.loc[
    babyland72_mean_shape_by_orientation['orientation'] == 'all_orientations_mixed',
    ['landmark_idx', 'anatomical_group', 'anatomical_label', 'valid_count', 'visibility_rate'],
].copy()
save_table(
    babyland72_validity_summary,
    OUTPUT_ROOT / 'babyland72' / 'tables' / 'landmark_visibility_and_valid_counts.csv',
)

babyland72_mean_shape_by_orientation.head()


,dataset,class_idx,orientation,landmark_idx,anatomical_group,anatomical_label,mean_x,mean_y,valid_count,visibility_rate
0,BabyLand72,None,all_orientations_mixed,0,face_contour,face_contour_1,0.482399,0.589185,111,0.356913
1,BabyLand72,None,all_orientations_mixed,1,face_contour,face_contour_2,0.515599,0.592824,111,0.356913
2,BabyLand72,None,all_orientations_mixed,2,face_contour,face_contour_3,0.545390,0.592655,110,0.353698
3,BabyLand72,None,all_orientations_mixed,3,face_contour,face_contour_4,0.573129,0.579602,106,0.340836
4,BabyLand72,None,all_orientations_mixed,4,face_contour,face_contour_5,0.585093,0.561796,111,0.356913


## Landmark variability analysis

This section computes per-landmark means, standard deviations, spatial variability, and BabyLand72 visibility rates.


In [71]:
def compute_landmark_variability(samples: list[NaturalDatasetSample], num_landmarks: int) -> pd.DataFrame:
    rows = []
    for landmark_idx in range(num_landmarks):
        xs, ys = [], []
        visibility_values = []
        for sample in samples:
            mask = valid_mask(sample, num_landmarks=num_landmarks)
            if landmark_idx < len(mask) and mask[landmark_idx]:
                x, y = sample.landmarks[landmark_idx]
                xs.append(float(x))
                ys.append(float(y))
            if sample.visibility is not None:
                visibility_values.append(int(sample.visibility[landmark_idx]))
        std_x = float(np.std(xs, ddof=1)) if len(xs) >= 2 else np.nan
        std_y = float(np.std(ys, ddof=1)) if len(ys) >= 2 else np.nan
        rows.append({
            'landmark_idx': landmark_idx,
            'anatomical_group': get_landmark_anatomical_group(landmark_idx),
            'anatomical_label': get_landmark_anatomical_label(landmark_idx),
            'valid_sample_count': len(xs),
            'mean_x': float(np.mean(xs)) if xs else np.nan,
            'mean_y': float(np.mean(ys)) if ys else np.nan,
            'std_x': std_x,
            'std_y': std_y,
            'spatial_std': float(np.hypot(std_x, std_y)) if np.isfinite(std_x) and np.isfinite(std_y) else np.nan,
            'visibility_rate': float(np.mean(visibility_values)) if visibility_values else np.nan,
        })
    return pd.DataFrame(rows)


def plot_spatial_std(variability_df: pd.DataFrame, title: str, output_path: Path, unit_label: str) -> None:
    fig, axes = plt.subplots(1, 2, figsize=(15, 4.8), constrained_layout=True)
    axes[0].plot(variability_df['landmark_idx'], variability_df['spatial_std'], color='#355C7D', linewidth=1.4)
    axes[0].scatter(variability_df['landmark_idx'], variability_df['spatial_std'], s=20, color='#355C7D')
    axes[0].set_title(title)
    axes[0].set_xlabel('Landmark index')
    axes[0].set_ylabel(f'Spatial std ({unit_label})')
    group_summary = variability_df.groupby('anatomical_group', as_index=False)['spatial_std'].mean().sort_values('spatial_std', ascending=False)
    axes[1].bar(group_summary['anatomical_group'], group_summary['spatial_std'], color=[ANATOMICAL_GROUP_COLORS.get(group, ANATOMICAL_GROUP_COLORS['unknown']) for group in group_summary['anatomical_group']])
    axes[1].set_title(title.replace('spatial std', 'anatomical-group spatial std'))
    axes[1].set_xlabel('Anatomical group')
    axes[1].set_ylabel(f'Mean spatial std ({unit_label})')
    axes[1].tick_params(axis='x', rotation=40)
    save_figure(fig, output_path)


def plot_visibility_heatmap(visibility_df: pd.DataFrame, title: str, output_path: Path) -> None:
    heatmap_df = visibility_df.pivot(index='landmark_idx', columns='orientation', values='visibility_rate')
    heatmap_df = heatmap_df.reindex(columns=ORIENTATION_ORDER + ['all'], fill_value=np.nan)
    fig, ax = plt.subplots(figsize=(8, 11), constrained_layout=True)
    image = ax.imshow(heatmap_df.values, aspect='auto', cmap='viridis', vmin=0.0, vmax=1.0)
    ax.set_title(title)
    ax.set_xlabel('Orientation')
    ax.set_ylabel('Landmark index')
    ax.set_xticks(np.arange(len(heatmap_df.columns)))
    ax.set_xticklabels(heatmap_df.columns, rotation=30)
    fig.colorbar(image, ax=ax, label='Visibility rate')
    save_figure(fig, output_path)


baby_variability_72 = compute_landmark_variability(babyland72_samples, BABYLAND72_LANDMARK_COUNT)
baby_variability_68 = compute_landmark_variability(babyland72_samples_68, COMPARISON_LANDMARK_COUNT)
infant_variability_68 = compute_landmark_variability(infantface_samples_68, COMPARISON_LANDMARK_COUNT)

save_table(baby_variability_72, OUTPUT_ROOT / 'babyland72' / 'tables' / 'babyland72_landmark_variability_72.csv')
save_table(baby_variability_68, OUTPUT_ROOT / 'babyland72' / 'tables' / 'babyland72_landmark_variability_68.csv')
save_table(infant_variability_68, OUTPUT_ROOT / 'infantface' / 'tables' / 'infantface_landmark_variability_68.csv')

plot_spatial_std(
    baby_variability_72,
    'BabyLand72 spatial std by landmark (72)',
    OUTPUT_ROOT / 'babyland72' / 'figures' / 'spatial_std_72',
    unit_label='normalized units',
)
plot_spatial_std(
    baby_variability_68,
    'BabyLand72 spatial std by landmark (68)',
    OUTPUT_ROOT / 'babyland72' / 'figures' / 'spatial_std_68',
    unit_label='normalized units',
)
plot_spatial_std(
    infant_variability_68,
    'InfantFace spatial std by landmark (68)',
    OUTPUT_ROOT / 'infantface' / 'figures' / 'spatial_std_68',
    unit_label='pixels',
)
plot_visibility_heatmap(babyland72_visibility_summary, 'BabyLand72 visibility rate by landmark and orientation', OUTPUT_ROOT / 'babyland72' / 'figures' / 'visibility_rate_heatmap')

baby_variability_68.sort_values('spatial_std', ascending=False).head(10)


,landmark_idx,anatomical_group,anatomical_label,valid_sample_count,mean_x,mean_y,std_x,std_y,spatial_std,visibility_rate
8,8,face_contour,face_contour_9,293,0.570504,0.515170,0.184593,0.181544,0.258907,0.942122
9,9,face_contour,face_contour_10,223,0.553984,0.511139,0.171709,0.175235,0.245340,0.717042
10,10,face_contour,face_contour_11,190,0.548596,0.525147,0.157308,0.174958,0.235279,0.610932
56,56,outer_lip,outer_lip_57,249,0.546046,0.492614,0.155441,0.161890,0.224433,0.800643
57,57,outer_lip,outer_lip_58,242,0.552231,0.495945,0.156983,0.159893,0.224075,0.778135
58,58,outer_lip,outer_lip_59,253,0.562770,0.492579,0.149317,0.160672,0.219342,0.813505
7,7,face_contour,face_contour_8,224,0.591209,0.535919,0.163758,0.145752,0.219227,0.720257
11,11,face_contour,face_contour_12,140,0.519661,0.531852,0.140155,0.167607,0.218485,0.450161
55,55,outer_lip,outer_lip_56,223,0.540002,0.490505,0.149025,0.155049,0.215055,0.717042
66,66,inner_lip,inner_lip_67,230,0.544468,0.491701,0.142516,0.152211,0.208516,0.739550


## BabyLand72 PCA strategy

BabyLand72 contains invalid coordinates for landmarks with `v = 0`, so the notebook does **not** fit PCA on raw 72-dimensional coordinate vectors with missing landmarks left in place.

Chosen strategy:
- compute landmark visibility rates first,
- choose a **stable common subset** of landmarks whose visibility rate is above a configurable threshold,
- keep only **complete-case samples** for that selected subset,
- fit PCA on that subset only,
- report the chosen threshold, selected landmark set, and retained sample count.

Why this strategy:
- it avoids injecting artificial structure through imputation,
- it avoids treating invalid `0 0` coordinates as geometry,
- it keeps the PCA interpretation straightforward for exploratory analysis.

The notebook tries progressively looser thresholds until it finds a subset with enough complete-case samples. InfantFace does not need this filtering because all 68 landmarks are valid.


## Internal PCA analysis per dataset

The notebook fits PCA separately for each dataset and, when possible, separately for each orientation class. It compares two normalization modes:
- `center_scale`: remove translation and scale only.
- `procrustes`: iterative no-reflection generalized Procrustes alignment.


In [72]:
def normalize_shape_center_scale(shape: np.ndarray, eps: float = 1e-8) -> tuple[np.ndarray, dict[str, Any]]:
    centroid = shape.mean(axis=0)
    centered = shape - centroid
    rms = float(np.sqrt(np.mean(np.sum(centered ** 2, axis=1))))
    scale = max(rms, eps)
    normalized = centered / scale
    return normalized, {'centroid': centroid, 'scale': scale}


def estimate_similarity_transform(source: np.ndarray, target: np.ndarray, allow_reflection: bool = False, eps: float = 1e-8) -> tuple[np.ndarray, float, np.ndarray]:
    source_mean = source.mean(axis=0)
    target_mean = target.mean(axis=0)
    source_centered = source - source_mean
    target_centered = target - target_mean
    source_norm = math.sqrt(max(float(np.sum(source_centered ** 2)), eps))
    target_norm = math.sqrt(max(float(np.sum(target_centered ** 2)), eps))
    source_normalized = source_centered / source_norm
    target_normalized = target_centered / target_norm
    covariance = source_normalized.T @ target_normalized
    u_matrix, _, vh_matrix = np.linalg.svd(covariance, full_matrices=False)
    rotation = vh_matrix.T @ u_matrix.T
    if (not allow_reflection) and np.linalg.det(rotation) < 0:
        vh_matrix[-1, :] *= -1
        rotation = vh_matrix.T @ u_matrix.T
    scale = target_norm / max(source_norm, eps)
    translation = target_mean - scale * (source_mean @ rotation)
    return rotation, scale, translation


def apply_similarity_transform(shape: np.ndarray, rotation: np.ndarray, scale: float, translation: np.ndarray) -> np.ndarray:
    return scale * (shape @ rotation) + translation


def generalized_procrustes(shapes: np.ndarray, max_iterations: int = 50, tolerance: float = 1e-7) -> tuple[np.ndarray, np.ndarray]:
    aligned = np.zeros_like(shapes)
    reference, _ = normalize_shape_center_scale(shapes[0])
    previous = reference.copy()
    for _ in range(max_iterations):
        for index, shape in enumerate(shapes):
            normalized, _ = normalize_shape_center_scale(shape)
            rotation, scale, translation = estimate_similarity_transform(normalized, previous, allow_reflection=False)
            aligned[index] = apply_similarity_transform(normalized, rotation, scale, translation)
        mean_shape = aligned.mean(axis=0)
        mean_shape, _ = normalize_shape_center_scale(mean_shape)
        delta = float(np.linalg.norm(mean_shape - previous))
        previous = mean_shape
        if delta < tolerance:
            break
    for index, shape in enumerate(shapes):
        normalized, _ = normalize_shape_center_scale(shape)
        rotation, scale, translation = estimate_similarity_transform(normalized, previous, allow_reflection=False)
        aligned[index] = apply_similarity_transform(normalized, rotation, scale, translation)
    return aligned, previous


def fit_pca(data_matrix: np.ndarray) -> dict[str, Any]:
    mean_vector = data_matrix.mean(axis=0)
    centered = data_matrix - mean_vector
    sample_count = data_matrix.shape[0]
    u_matrix, singular_values, vh_matrix = np.linalg.svd(centered, full_matrices=False)
    eigenvalues = (singular_values ** 2) / max(sample_count - 1, 1)
    explained_variance_ratio = eigenvalues / max(eigenvalues.sum(), 1e-12)
    scores = centered @ vh_matrix.T
    return {
        'mean_vector': mean_vector,
        'components': vh_matrix,
        'explained_variance': eigenvalues,
        'explained_variance_ratio': explained_variance_ratio,
        'scores': scores,
    }


def components_for_variance(explained_variance_ratio: np.ndarray, threshold: float) -> int:
    cumulative = np.cumsum(explained_variance_ratio)
    return int(np.searchsorted(cumulative, threshold) + 1)


def compute_truncated_reconstruction_metrics(
    data_matrix: np.ndarray,
    mean_vector: np.ndarray,
    components: np.ndarray,
    explained_variance_ratio: np.ndarray,
    variance_target: float = PCA_RECONSTRUCTION_VARIANCE_TARGET,
) -> dict[str, Any]:
    component_count = min(components_for_variance(explained_variance_ratio, variance_target), components.shape[0])
    centered = data_matrix - mean_vector
    truncated_components = components[:component_count]
    truncated_scores = centered @ truncated_components.T
    truncated_reconstruction = mean_vector + truncated_scores @ truncated_components
    pointwise_error = (data_matrix - truncated_reconstruction) ** 2
    reconstruction_error = np.mean(pointwise_error, axis=1)
    return {
        'reconstruction': truncated_reconstruction,
        'reconstruction_error': reconstruction_error,
        'reconstruction_components_used': component_count,
        'reconstruction_variance_target': variance_target,
    }


def flatten_shapes(shapes: np.ndarray) -> np.ndarray:
    return shapes.reshape(shapes.shape[0], -1)


def build_complete_case_matrix(samples: list[NaturalDatasetSample], landmark_indices: list[int]) -> tuple[np.ndarray, list[NaturalDatasetSample]]:
    kept_shapes = []
    kept_samples = []
    for sample in samples:
        mask = valid_mask(sample)
        if all(mask[index] for index in landmark_indices):
            kept_shapes.append(sample.landmarks[landmark_indices].astype(np.float64, copy=True))
            kept_samples.append(sample)
    if not kept_shapes:
        return np.empty((0, len(landmark_indices), 2), dtype=np.float64), []
    return np.stack(kept_shapes, axis=0), kept_samples


def choose_babyland72_pca_subset(samples: list[NaturalDatasetSample], candidate_indices: list[int], min_samples: int) -> dict[str, Any]:
    visibility_rates = {}
    for landmark_idx in candidate_indices:
        values = np.asarray([sample.visibility[landmark_idx] for sample in samples], dtype=np.float64)
        visibility_rates[landmark_idx] = float(values.mean())
    for threshold in BABYLAND72_PCA_CANDIDATE_THRESHOLDS:
        selected = [index for index in candidate_indices if visibility_rates[index] >= threshold]
        if len(selected) < BABYLAND72_PCA_MIN_SELECTED_LANDMARKS:
            continue
        shape_matrix, kept_samples = build_complete_case_matrix(samples, selected)
        if len(kept_samples) >= min_samples:
            return {
                'selected_landmarks': selected,
                'visibility_threshold': threshold,
                'shape_matrix': shape_matrix,
                'kept_samples': kept_samples,
                'visibility_rates': visibility_rates,
            }
    fallback_selected = [index for index in candidate_indices if visibility_rates[index] >= min(BABYLAND72_PCA_CANDIDATE_THRESHOLDS)]
    shape_matrix, kept_samples = build_complete_case_matrix(samples, fallback_selected)
    return {
        'selected_landmarks': fallback_selected,
        'visibility_threshold': min(BABYLAND72_PCA_CANDIDATE_THRESHOLDS),
        'shape_matrix': shape_matrix,
        'kept_samples': kept_samples,
        'visibility_rates': visibility_rates,
    }


def prepare_shapes_for_pca(samples: list[NaturalDatasetSample], landmark_indices: list[int], alignment_method: str) -> tuple[np.ndarray, np.ndarray | None]:
    shape_matrix, kept_samples = build_complete_case_matrix(samples, landmark_indices)
    if len(kept_samples) == 0:
        return np.empty((0, len(landmark_indices), 2), dtype=np.float64), None
    if alignment_method == 'center_scale':
        normalized = np.stack([normalize_shape_center_scale(shape)[0] for shape in shape_matrix], axis=0)
        return normalized, None
    if alignment_method == 'procrustes':
        aligned, reference = generalized_procrustes(shape_matrix)
        return aligned, reference
    raise ValueError(f'Unknown alignment method: {alignment_method}')


def fit_dataset_pca(samples: list[NaturalDatasetSample], dataset_name: str, landmark_indices: list[int], alignment_method: str, scope_name: str) -> dict[str, Any] | None:
    prepared_shapes, reference_shape = prepare_shapes_for_pca(samples, landmark_indices)
    if prepared_shapes.shape[0] < 3:
        return None
    data_matrix = flatten_shapes(prepared_shapes)
    pca = fit_pca(data_matrix)
    pca.update(compute_truncated_reconstruction_metrics(
        data_matrix=data_matrix,
        mean_vector=pca['mean_vector'],
        components=pca['components'],
        explained_variance_ratio=pca['explained_variance_ratio'],
        variance_target=PCA_RECONSTRUCTION_VARIANCE_TARGET,
    ))
    pca.update({
        'dataset_name': dataset_name,
        'scope_name': scope_name,
        'alignment_method': alignment_method,
        'sample_count': prepared_shapes.shape[0],
        'num_landmarks': len(landmark_indices),
        'landmark_indices': landmark_indices,
        'prepared_shapes': prepared_shapes,
        'reference_shape': reference_shape,
        'sample_ids': [sample.image_id for sample in samples if all(valid_mask(sample)[index] for index in landmark_indices)],
        'class_indices': [sample.class_idx for sample in samples if all(valid_mask(sample)[index] for index in landmark_indices)],
        'orientations': [sample.orientation for sample in samples if all(valid_mask(sample)[index] for index in landmark_indices)],
    })
    return pca


In [75]:
def build_pca_experiments() -> tuple[dict[str, Any], pd.DataFrame]:
    experiments: dict[str, Any] = {'BabyLand72': {}, 'InfantFace': {}}
    strategy_rows = []

    baby_68_subset = choose_babyland72_pca_subset(
        babyland72_samples_68,
        list(range(COMPARISON_LANDMARK_COUNT)),
        min_samples=BABYLAND72_PCA_MIN_GLOBAL_SAMPLES,
    )
    strategy_rows.append({
        'dataset': 'BabyLand72',
        'landmark_scope': '68',
        'selected_landmark_count': len(baby_68_subset['selected_landmarks']),
        'selected_landmark_indices': ' '.join(map(str, baby_68_subset['selected_landmarks'])),
        'visibility_threshold': baby_68_subset['visibility_threshold'],
        'complete_case_sample_count': len(baby_68_subset['kept_samples']),
    })
    baby_72_subset = choose_babyland72_pca_subset(
        babyland72_samples,
        list(range(BABYLAND72_LANDMARK_COUNT)),
        min_samples=BABYLAND72_PCA_MIN_GLOBAL_SAMPLES,
    )
    strategy_rows.append({
        'dataset': 'BabyLand72',
        'landmark_scope': '72',
        'selected_landmark_count': len(baby_72_subset['selected_landmarks']),
        'selected_landmark_indices': ' '.join(map(str, baby_72_subset['selected_landmarks'])),
        'visibility_threshold': baby_72_subset['visibility_threshold'],
        'complete_case_sample_count': len(baby_72_subset['kept_samples']),
    })

    experiments['BabyLand72']['subset_68'] = baby_68_subset
    experiments['BabyLand72']['subset_72'] = baby_72_subset

    for alignment_method in ('center_scale', 'procrustes'):
        experiments['BabyLand72'][f'global_68_{alignment_method}'] = fit_dataset_pca(
            baby_68_subset['kept_samples'],
            'BabyLand72',
            baby_68_subset['selected_landmarks'],
            alignment_method,
            scope_name='global_68',
        )
        experiments['BabyLand72'][f'global_72_{alignment_method}'] = fit_dataset_pca(
            baby_72_subset['kept_samples'],
            'BabyLand72',
            baby_72_subset['selected_landmarks'],
            alignment_method,
            scope_name='global_72',
        )
        experiments['InfantFace'][f'global_68_{alignment_method}'] = fit_dataset_pca(
            infantface_samples_68,
            'InfantFace',
            list(range(COMPARISON_LANDMARK_COUNT)),
            alignment_method,
            scope_name='global_68',
        )

    for dataset_name, samples, subset_key in [
        ('BabyLand72', baby_68_subset['kept_samples'], baby_68_subset['selected_landmarks']),
        ('InfantFace', infantface_samples_68, list(range(COMPARISON_LANDMARK_COUNT))),
    ]:
        for class_idx, orientation in CLASS_ID_TO_NAME.items():
            subset_samples = [sample for sample in samples if sample.class_idx == class_idx]
            min_samples = BABYLAND72_PCA_MIN_CLASS_SAMPLES if dataset_name == 'BabyLand72' else 6
            if len(subset_samples) < min_samples:
                strategy_rows.append({
                    'dataset': dataset_name,
                    'landmark_scope': '68',
                    'selected_landmark_count': len(subset_key),
                    'selected_landmark_indices': ' '.join(map(str, subset_key)),
                    'visibility_threshold': np.nan,
                    'complete_case_sample_count': len(subset_samples),
                    'class_idx': class_idx,
                    'orientation': orientation,
                    'status': 'skipped_insufficient_samples',
                })
                continue
            for alignment_method in ('center_scale', 'procrustes'):
                experiments[dataset_name][f'class_{class_idx}_68_{alignment_method}'] = fit_dataset_pca(
                    subset_samples,
                    dataset_name,
                    subset_key,
                    alignment_method,
                    scope_name=f'class_{class_idx}_68',
                )
    return experiments, pd.DataFrame(strategy_rows)


def fit_dataset_pca(samples: list[NaturalDatasetSample], dataset_name: str, landmark_indices: list[int], alignment_method: str, scope_name: str) -> dict[str, Any] | None:
    prepared_shapes, reference_shape = prepare_shapes_for_pca(samples, landmark_indices, alignment_method)
    if prepared_shapes.shape[0] < 3:
        return None
    data_matrix = flatten_shapes(prepared_shapes)
    pca = fit_pca(data_matrix)
    pca.update(compute_truncated_reconstruction_metrics(
        data_matrix=data_matrix,
        mean_vector=pca['mean_vector'],
        components=pca['components'],
        explained_variance_ratio=pca['explained_variance_ratio'],
        variance_target=PCA_RECONSTRUCTION_VARIANCE_TARGET,
    ))
    pca.update({
        'dataset_name': dataset_name,
        'scope_name': scope_name,
        'alignment_method': alignment_method,
        'sample_count': prepared_shapes.shape[0],
        'num_landmarks': len(landmark_indices),
        'landmark_indices': landmark_indices,
        'prepared_shapes': prepared_shapes,
        'reference_shape': reference_shape,
        'sample_ids': [sample.image_id for sample in samples if all(valid_mask(sample)[index] for index in landmark_indices)],
        'class_indices': [sample.class_idx for sample in samples if all(valid_mask(sample)[index] for index in landmark_indices)],
        'orientations': [sample.orientation for sample in samples if all(valid_mask(sample)[index] for index in landmark_indices)],
    })
    return pca
save_table(babyland72_pca_strategy, OUTPUT_ROOT / 'babyland72' / 'tables' / 'babyland72_pca_strategy.csv')
babyland72_pca_strategy.head()


,dataset,landmark_scope,selected_landmark_count,visibility_threshold,complete_case_sample_count
0,BabyLand72,68,11,0.8,173
1,BabyLand72,72,15,0.8,119


## PCA explained variance

This section reports explained-variance patterns and the number of components needed to reach common variance thresholds.


In [76]:
def explained_variance_summary(pca_result: dict[str, Any]) -> pd.DataFrame:
    cumulative = np.cumsum(pca_result['explained_variance_ratio'])
    thresholds = [0.90, 0.95, 0.98]
    row = {
        'dataset': pca_result['dataset_name'],
        'scope_name': pca_result['scope_name'],
        'alignment_method': pca_result['alignment_method'],
        'sample_count': pca_result['sample_count'],
        'num_landmarks': pca_result['num_landmarks'],
        'reconstruction_components_used': pca_result.get('reconstruction_components_used', np.nan),
        'reconstruction_variance_target': pca_result.get('reconstruction_variance_target', np.nan),
    }
    for threshold in thresholds:
        row[f'components_for_{int(threshold * 100)}pct'] = int(np.searchsorted(cumulative, threshold) + 1)
    return pd.DataFrame([row])


def plot_explained_variance(pca_result: dict[str, Any], output_path: Path) -> None:
    ratio = pca_result['explained_variance_ratio']
    cumulative = np.cumsum(ratio)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)
    component_indices = np.arange(1, len(ratio) + 1)
    axes[0].plot(component_indices, ratio, marker='o', linewidth=1.1, markersize=3.0)
    axes[0].set_title(f"{pca_result['dataset_name']} | {pca_result['scope_name']} | {pca_result['alignment_method']}\nExplained variance ratio")
    axes[0].set_xlabel('Principal component')
    axes[0].set_ylabel('Explained variance ratio')
    axes[1].plot(component_indices, cumulative, marker='o', linewidth=1.1, markersize=3.0)
    axes[1].axhline(0.90, color='#888888', linestyle='--', linewidth=1)
    axes[1].axhline(0.95, color='#666666', linestyle='--', linewidth=1)
    axes[1].axhline(0.98, color='#444444', linestyle='--', linewidth=1)
    axes[1].axvline(pca_result.get('reconstruction_components_used', 0), color='#aa3377', linestyle=':', linewidth=1.2)
    axes[1].set_title('Cumulative explained variance')
    axes[1].set_xlabel('Principal component')
    axes[1].set_ylabel('Cumulative ratio')
    save_figure(fig, output_path)


variance_rows = []
for dataset_name, experiments in pca_experiments.items():
    for key, pca_result in experiments.items():
        if not is_fitted_pca_result(pca_result):
            continue
        variance_rows.append(explained_variance_summary(pca_result))
        dataset_dir = OUTPUT_ROOT / dataset_name.lower() / 'figures'
        plot_explained_variance(pca_result, dataset_dir / f'{key}_explained_variance')
variance_summary_df = pd.concat(variance_rows, ignore_index=True) if variance_rows else pd.DataFrame()
save_table(variance_summary_df, OUTPUT_ROOT / 'tables' / 'pca_explained_variance_summary.csv')
variance_summary_df.head()


,dataset,scope_name,alignment_method,sample_count,num_landmarks,reconstruction_components_used,reconstruction_variance_target,components_for_90pct,components_for_95pct,components_for_98pct
0,BabyLand72,global_68,center_scale,173,11,NaN,NaN,2,3,4
1,BabyLand72,global_72,center_scale,119,15,NaN,NaN,2,2,4
2,BabyLand72,global_68,procrustes,173,11,NaN,NaN,2,2,4
3,BabyLand72,global_72,procrustes,119,15,NaN,NaN,2,2,3
4,BabyLand72,class_0_68,center_scale,17,11,NaN,NaN,1,2,3


## PCA component visualization

These plots visualize the mean shape and the deformation implied by each principal component.


In [77]:
def vector_to_shape(vector: np.ndarray, num_landmarks: int) -> np.ndarray:
    return vector.reshape(num_landmarks, 2)


def plot_pca_mode_triplet(
    pca_result: dict[str, Any],
    output_dir: Path,
    num_components_to_plot: int = PCA_COMPONENTS_TO_PLOT,
    alpha: float = 2.0,
) -> None:
    num_components = min(num_components_to_plot, len(pca_result['explained_variance']))
    mean_shape = vector_to_shape(pca_result['mean_vector'], pca_result['num_landmarks'])
    valid_mask_mean = np.isfinite(mean_shape).all(axis=1)
    landmark_indices = list(pca_result.get('landmark_indices', range(pca_result['num_landmarks'])))
    selected_label = f"selected landmarks = {len(landmark_indices)}"
    for component_idx in range(num_components):
        component = pca_result['components'][component_idx]
        eigenvalue = pca_result['explained_variance'][component_idx]
        delta = alpha * np.sqrt(max(float(eigenvalue), 0.0)) * component
        minus_shape = vector_to_shape(pca_result['mean_vector'] - delta, pca_result['num_landmarks'])
        plus_shape = vector_to_shape(pca_result['mean_vector'] + delta, pca_result['num_landmarks'])
        fig, axes = plt.subplots(1, 3, figsize=(13.6, 4.6), sharex=True, sharey=True, constrained_layout=True)
        panels = [
            (minus_shape, '-alpha', '#1f77b4'),
            (mean_shape, 'mean', '#333333'),
            (plus_shape, '+alpha', '#d62728'),
        ]
        for axis, (shape, subtitle, color) in zip(axes, panels):
            draw_shape_panel(
                ax=axis,
                shape=shape,
                title=subtitle,
                num_landmarks=pca_result['num_landmarks'],
                dataset_name=pca_result['dataset_name'],
                coordinate_space='normalized',
                valid_landmark_mask=valid_mask_mean,
                point_color=color,
                line_color=color,
                landmark_indices=landmark_indices,
                point_size=18,
                line_width=1.45,
            )
        apply_shared_shape_limits(axes, [minus_shape, mean_shape, plus_shape])
        fig.suptitle(
            f"PC{component_idx + 1} | explained variance ratio = {pca_result['explained_variance_ratio'][component_idx]:.3f}\n{selected_label}"
        )
        save_figure(fig, output_dir / f"{pca_result['scope_name']}_{pca_result['alignment_method']}_pc{component_idx + 1:02d}_mode_triplet")


for dataset_name, experiments in pca_experiments.items():
    for key, pca_result in experiments.items():
        if not is_fitted_pca_result(pca_result):
            continue
        plot_pca_mode_triplet(
            pca_result,
            OUTPUT_ROOT / dataset_name.lower() / 'figures',
        )


## PCA score-space analysis

These plots show how samples distribute in score space and whether global PCA is dominated by orientation.


In [78]:
def plot_score_scatter(pca_result: dict[str, Any], component_pairs: list[tuple[int, int]], output_path: Path) -> None:
    scores = pca_result['scores']
    if scores.shape[1] < 2:
        return
    fig, axes = plt.subplots(1, len(component_pairs), figsize=(5.2 * len(component_pairs), 4.5), constrained_layout=True)
    axes = np.atleast_1d(axes)
    class_indices = np.asarray(pca_result['class_indices'])
    for axis, (first_idx, second_idx) in zip(axes, component_pairs):
        if max(first_idx, second_idx) >= scores.shape[1]:
            axis.set_axis_off()
            continue
        for class_idx, orientation in CLASS_ID_TO_NAME.items():
            mask = class_indices == class_idx
            if not mask.any():
                continue
            axis.scatter(scores[mask, first_idx], scores[mask, second_idx], s=18, alpha=0.72, color=ORIENTATION_COLORS[orientation], label=orientation)
        axis.set_xlabel(f'PC{first_idx + 1} score')
        axis.set_ylabel(f'PC{second_idx + 1} score')
        axis.set_title(f'PC{first_idx + 1} vs PC{second_idx + 1}')
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, frameon=False, bbox_to_anchor=(1.02, 1.0), loc='upper left')
    fig.suptitle(f"{pca_result['dataset_name']} | {pca_result['scope_name']} | {pca_result['alignment_method']}")
    save_figure(fig, output_path)


for dataset_name, experiments in pca_experiments.items():
    for key, pca_result in experiments.items():
        if not is_fitted_pca_result(pca_result):
            continue
        plot_score_scatter(
            pca_result,
            component_pairs=[(0, 1), (0, 2), (1, 2)],
            output_path=OUTPUT_ROOT / dataset_name.lower() / 'figures' / f'{key}_score_scatter',
        )


## Normality analysis of PCA scores

The goal here is to assess whether the score distributions look approximately Gaussian or whether they appear skewed, heavy-tailed, or multimodal.


In [79]:
def normality_statistics(pca_result: dict[str, Any], max_components: int = 10) -> pd.DataFrame:
    rows = []
    scores = pca_result['scores']
    num_components = min(max_components, scores.shape[1])
    for component_idx in range(num_components):
        values = scores[:, component_idx]
        mean = float(np.mean(values))
        std = float(np.std(values, ddof=1)) if len(values) >= 2 else np.nan
        z_scores = np.zeros_like(values) if not np.isfinite(std) or std <= 1e-12 else (values - mean) / std
        outlier_rate = float(np.mean(np.abs(z_scores) > 3.0))
        skewness = float(stats.skew(values, bias=False)) if stats is not None and len(values) >= 3 else np.nan
        kurtosis = float(stats.kurtosis(values, fisher=True, bias=False)) if stats is not None and len(values) >= 4 else np.nan
        shapiro_p = np.nan
        dagostino_p = np.nan
        if stats is not None and 3 <= len(values) <= 5000:
            try:
                shapiro_p = float(stats.shapiro(values).pvalue)
            except Exception:
                shapiro_p = np.nan
        if stats is not None and len(values) >= 8:
            try:
                dagostino_p = float(stats.normaltest(values).pvalue)
            except Exception:
                dagostino_p = np.nan
        rows.append({
            'dataset': pca_result['dataset_name'],
            'scope_name': pca_result['scope_name'],
            'alignment_method': pca_result['alignment_method'],
            'component': component_idx + 1,
            'mean': mean,
            'std': std,
            'skewness': skewness,
            'excess_kurtosis': kurtosis,
            'shapiro_pvalue': shapiro_p,
            'dagostino_pvalue': dagostino_p,
            'zscore_outlier_rate_gt_3': outlier_rate,
        })
    return pd.DataFrame(rows)


def plot_score_distribution_diagnostics(pca_result: dict[str, Any], components: tuple[int, ...] = (0, 1, 2), output_path: Path | None = None) -> None:
    fig, axes = plt.subplots(len(components), 2, figsize=(11, 3.6 * len(components)), constrained_layout=True)
    axes = np.atleast_2d(axes)
    for row_index, component_idx in enumerate(components):
        if component_idx >= pca_result['scores'].shape[1]:
            axes[row_index, 0].set_axis_off()
            axes[row_index, 1].set_axis_off()
            continue
        values = pca_result['scores'][:, component_idx]
        axes[row_index, 0].hist(values, bins=24, color='#4E79A7', alpha=0.85, density=True)
        axes[row_index, 0].set_title(f'PC{component_idx + 1} histogram')
        axes[row_index, 0].set_xlabel('Score')
        axes[row_index, 0].set_ylabel('Density')
        if stats is not None and len(values) >= 3:
            qq = stats.probplot(values, dist='norm')
            theoretical, ordered = qq[0]
            slope, intercept, _ = qq[1]
            axes[row_index, 1].scatter(theoretical, ordered, s=12, alpha=0.75, color='#C06C84')
            axes[row_index, 1].plot(theoretical, slope * theoretical + intercept, color='#444444', linewidth=1.0)
            axes[row_index, 1].set_title(f'PC{component_idx + 1} Q-Q plot')
            axes[row_index, 1].set_xlabel('Theoretical quantiles')
            axes[row_index, 1].set_ylabel('Observed quantiles')
        else:
            axes[row_index, 1].set_axis_off()
    fig.suptitle(f"{pca_result['dataset_name']} | {pca_result['scope_name']} | {pca_result['alignment_method']}")
    if output_path is not None:
        save_figure(fig, output_path)


normality_tables = []
for dataset_name, experiments in pca_experiments.items():
    for key, pca_result in experiments.items():
        if not is_fitted_pca_result(pca_result):
            continue
        normality_tables.append(normality_statistics(pca_result))
        plot_score_distribution_diagnostics(pca_result, output_path=OUTPUT_ROOT / dataset_name.lower() / 'figures' / f'{key}_score_diagnostics')
normality_df = pd.concat(normality_tables, ignore_index=True) if normality_tables else pd.DataFrame()
save_table(normality_df, OUTPUT_ROOT / 'tables' / 'pca_score_normality_statistics.csv')
normality_df.head()


/Users/jocareher/anaconda3/envs/dev/lib/python3.9/site-packages/scipy/stats/_stats_py.py:1971: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=17
  k, _ = kurtosistest(a, axis)
/Users/jocareher/anaconda3/envs/dev/lib/python3.9/site-packages/scipy/stats/_stats_py.py:1971: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=17
  k, _ = kurtosistest(a, axis)
/Users/jocareher/anaconda3/envs/dev/lib/python3.9/site-packages/scipy/stats/_stats_py.py:1971: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=18
  k, _ = kurtosistest(a, axis)
/Users/jocareher/anaconda3/envs/dev/lib/python3.9/site-packages/scipy/stats/_stats_py.py:1971: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=18
  k, _ = kurtosistest(a, axis)
/Users/jocareher/anaconda3/envs/dev/lib/python3.9/site-packages/scipy/stats/_stats_py.py:1971: UserWarning: kurtosistest only valid for n>=20 ... continuing anyway, n=13
  k, _ = kurtosistest(a, a

,dataset,scope_name,alignment_method,component,mean,std,skewness,excess_kurtosis,shapiro_pvalue,dagostino_pvalue,zscore_outlier_rate_gt_3
0,BabyLand72,global_68,center_scale,1,1.026796e-17,2.331174,-0.269983,-1.617599,6.080190e-13,0.000000,0.00000
1,BabyLand72,global_68,center_scale,2,1.418262e-16,1.281005,-0.965832,0.308961,2.821677e-10,0.000015,0.00000
2,BabyLand72,global_68,center_scale,3,-8.856114e-17,0.438266,-0.569591,-0.634841,1.818982e-06,0.000709,0.00000
3,BabyLand72,global_68,center_scale,4,-1.495271e-16,0.298192,-0.024506,-0.165079,4.203104e-01,0.933021,0.00578
4,BabyLand72,global_68,center_scale,5,-2.823689e-17,0.214159,0.139449,0.247946,9.325183e-01,0.544874,0.00578


## Pairwise analysis

This section includes pairwise PCA score plots, class-distance heatmaps, and simple landmark-coordinate correlation summaries.


In [80]:
def pairwise_class_distance_matrix(pca_result: dict[str, Any]) -> pd.DataFrame:
    shapes = pca_result['prepared_shapes']
    class_indices = np.asarray(pca_result['class_indices'])
    rows = []
    for class_i, orientation_i in CLASS_ID_TO_NAME.items():
        for class_j, orientation_j in CLASS_ID_TO_NAME.items():
            subset_i = shapes[class_indices == class_i]
            subset_j = shapes[class_indices == class_j]
            if len(subset_i) == 0 or len(subset_j) == 0:
                value = np.nan
            else:
                distances = []
                for shape_i in subset_i:
                    flattened_i = shape_i.reshape(-1)
                    flattened_j = subset_j.reshape(len(subset_j), -1)
                    distance_values = np.linalg.norm(flattened_j - flattened_i[None, :], axis=1)
                    distances.extend(distance_values.tolist())
                value = float(np.mean(distances)) if distances else np.nan
            rows.append({
                'class_i': class_i,
                'orientation_i': orientation_i,
                'class_j': class_j,
                'orientation_j': orientation_j,
                'mean_distance': value,
            })
    return pd.DataFrame(rows)


def plot_distance_heatmap(distance_df: pd.DataFrame, title: str, output_path: Path) -> None:
    pivot = distance_df.pivot(index='orientation_i', columns='orientation_j', values='mean_distance')
    pivot = pivot.reindex(index=ORIENTATION_ORDER, columns=ORIENTATION_ORDER)
    fig, ax = plt.subplots(figsize=(7, 6), constrained_layout=True)
    image = ax.imshow(pivot.values, cmap='magma')
    ax.set_title(title)
    ax.set_xticks(np.arange(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns, rotation=30)
    ax.set_yticks(np.arange(len(pivot.index)))
    ax.set_yticklabels(pivot.index)
    for row_idx in range(len(pivot.index)):
        for col_idx in range(len(pivot.columns)):
            value = pivot.values[row_idx, col_idx]
            if np.isfinite(value):
                ax.text(col_idx, row_idx, f'{value:.2f}', ha='center', va='center', color='white', fontsize=9)
    fig.colorbar(image, ax=ax, label='Mean pairwise distance')
    save_figure(fig, output_path)


def landmark_coordinate_correlation(variability_df: pd.DataFrame) -> pd.DataFrame:
    summary = variability_df.groupby('anatomical_group', as_index=False)['spatial_std'].mean().rename(columns={'spatial_std': 'mean_spatial_std'})
    return summary.sort_values('mean_spatial_std', ascending=False)


pairwise_distance_tables = []
for dataset_name, experiments in pca_experiments.items():
    for key, pca_result in experiments.items():
        if not is_fitted_pca_result(pca_result) or not key.startswith('global_68_'):
            continue
        distance_df = pairwise_class_distance_matrix(pca_result)
        pairwise_distance_tables.append(distance_df.assign(dataset=dataset_name, key=key, alignment_method=pca_result['alignment_method']))
        save_table(distance_df, OUTPUT_ROOT / dataset_name.lower() / 'tables' / f'{key}_pairwise_class_distances.csv')
        plot_distance_heatmap(distance_df, f"{dataset_name} | {pca_result['alignment_method']} | class distance matrix", OUTPUT_ROOT / dataset_name.lower() / 'figures' / f'{key}_pairwise_class_distances')

correlation_summary_baby = landmark_coordinate_correlation(baby_variability_68)
correlation_summary_infant = landmark_coordinate_correlation(infant_variability_68)
save_table(correlation_summary_baby, OUTPUT_ROOT / 'babyland72' / 'tables' / 'babyland72_anatomical_group_variability.csv')
save_table(correlation_summary_infant, OUTPUT_ROOT / 'infantface' / 'tables' / 'infantface_anatomical_group_variability.csv')


PosixPath('/Users/jocareher/Library/CloudStorage/OneDrive-Personal/Educacion/PhD_UPF_2023/landmarks_detection/natural_individual_shape_analysis_outputs/infantface/tables/infantface_anatomical_group_variability.csv')

## Reconstruction error analysis

This section compares each dataset's own global PCA and class-conditioned PCA reconstruction behavior.


In [81]:
def build_reconstruction_error_table(dataset_name: str, experiments: dict[str, Any]) -> pd.DataFrame:
    rows = []
    for key, pca_result in experiments.items():
        if not is_fitted_pca_result(pca_result):
            continue
        for image_id, class_idx, orientation, error_value in zip(
            pca_result['sample_ids'],
            pca_result['class_indices'],
            pca_result['orientations'],
            pca_result['reconstruction_error'],
        ):
            rows.append({
                'dataset': dataset_name,
                'experiment_key': key,
                'scope_name': pca_result['scope_name'],
                'alignment_method': pca_result['alignment_method'],
                'image_id': image_id,
                'class_idx': class_idx,
                'orientation': orientation,
                'reconstruction_error': float(error_value),
                'reconstruction_components_used': int(pca_result.get('reconstruction_components_used', 0)),
                'reconstruction_variance_target': float(pca_result.get('reconstruction_variance_target', np.nan)),
            })
    return pd.DataFrame(rows)


def plot_reconstruction_boxplots(error_df: pd.DataFrame, dataset_name: str, output_path: Path) -> None:
    if error_df.empty:
        return
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.8), constrained_layout=True)
    scope_order = sorted(error_df['experiment_key'].unique())
    data_by_scope = [error_df.loc[error_df['experiment_key'] == scope, 'reconstruction_error'].values for scope in scope_order]
    axes[0].boxplot(data_by_scope, labels=scope_order, showfliers=False)
    axes[0].set_title(f'{dataset_name}: truncated reconstruction error by PCA experiment')
    axes[0].set_ylabel('Reconstruction MSE')
    axes[0].tick_params(axis='x', rotation=45)

    grouped = [error_df.loc[error_df['orientation'] == orientation, 'reconstruction_error'].values for orientation in ORIENTATION_ORDER if (error_df['orientation'] == orientation).any()]
    group_labels = [orientation for orientation in ORIENTATION_ORDER if (error_df['orientation'] == orientation).any()]
    axes[1].boxplot(grouped, labels=group_labels, showfliers=False)
    axes[1].set_title(f'{dataset_name}: truncated reconstruction error by orientation')
    axes[1].set_ylabel('Reconstruction MSE')
    axes[1].tick_params(axis='x', rotation=25)
    save_figure(fig, output_path)


baby_reconstruction_errors = build_reconstruction_error_table('BabyLand72', pca_experiments['BabyLand72'])
infant_reconstruction_errors = build_reconstruction_error_table('InfantFace', pca_experiments['InfantFace'])
save_table(baby_reconstruction_errors, OUTPUT_ROOT / 'babyland72' / 'tables' / 'babyland72_reconstruction_errors.csv')
save_table(infant_reconstruction_errors, OUTPUT_ROOT / 'infantface' / 'tables' / 'infantface_reconstruction_errors.csv')
plot_reconstruction_boxplots(baby_reconstruction_errors, 'BabyLand72', OUTPUT_ROOT / 'babyland72' / 'figures' / 'reconstruction_errors')
plot_reconstruction_boxplots(infant_reconstruction_errors, 'InfantFace', OUTPUT_ROOT / 'infantface' / 'figures' / 'reconstruction_errors')

baby_reconstruction_errors.head()


,dataset,experiment_key,scope_name,alignment_method,image_id,class_idx,orientation,reconstruction_error,reconstruction_components_used,reconstruction_variance_target
0,BabyLand72,global_68_center_scale,global_68,center_scale,face_bcn_02,1,quarter_left,8.136179e-32,0,NaN
1,BabyLand72,global_68_center_scale,global_68,center_scale,face_bcn_03,2,frontal,1.038470e-31,0,NaN
2,BabyLand72,global_68_center_scale,global_68,center_scale,face_bcn_05,1,quarter_left,7.155486e-32,0,NaN
3,BabyLand72,global_68_center_scale,global_68,center_scale,face_bcn_06,2,frontal,2.006745e-31,0,NaN
4,BabyLand72,global_68_center_scale,global_68,center_scale,face_bcn_07,2,frontal,1.601821e-31,0,NaN


## Outlier inspection

This section identifies the samples with the highest reconstruction error within each dataset and saves shape plots plus optional image overlays.


In [82]:
def draw_landmarks_on_image(sample: NaturalDatasetSample, output_path: Path, num_landmarks: int) -> Path | None:
    if sample.image_path is None or not sample.image_path.exists():
        return None
    image = Image.open(sample.image_path).convert('RGB')
    draw = ImageDraw.Draw(image)
    mask = valid_mask(sample, num_landmarks=num_landmarks)
    for landmark_idx, (x, y) in enumerate(sample.landmarks[:num_landmarks]):
        if not mask[landmark_idx]:
            continue
        group = get_landmark_anatomical_group(landmark_idx)
        color = ANATOMICAL_GROUP_COLORS.get(group, ANATOMICAL_GROUP_COLORS['unknown'])
        rgb = tuple(int(color[index:index + 2], 16) for index in (1, 3, 5))
        radius = 2
        draw.ellipse((x - radius, y - radius, x + radius, y + radius), fill=rgb, outline=rgb)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    image.save(output_path, quality=92, optimize=True)
    return output_path


def save_outlier_report(samples: list[NaturalDatasetSample], error_df: pd.DataFrame, dataset_name: str, num_landmarks: int) -> pd.DataFrame:
    if error_df.empty:
        return pd.DataFrame()
    top_df = error_df.sort_values('reconstruction_error', ascending=False).head(OUTLIER_TOP_K).copy()
    sample_by_id = {sample.image_id: sample for sample in samples}
    rows = []
    for _, row in top_df.iterrows():
        sample = sample_by_id.get(row['image_id'])
        if sample is None:
            continue
        outlier_dir = OUTPUT_ROOT / dataset_name.lower() / 'outliers'
        fig, ax = plt.subplots(figsize=(6, 6), constrained_layout=True)
        plot_shape(
            ax,
            sample.landmarks[:num_landmarks],
            f"{dataset_name}: {sample.image_id}\nerror={row['reconstruction_error']:.4f}",
            num_landmarks,
            dataset_name=dataset_name,
            coordinate_space='raw',
            valid_landmark_mask=valid_mask(sample, num_landmarks=num_landmarks),
        )
        shape_plot_path = outlier_dir / f"{sample.image_id}_{row['experiment_key']}_shape"
        save_figure(fig, shape_plot_path)
        overlay_path = draw_landmarks_on_image(sample, outlier_dir / f"{sample.image_id}_{row['experiment_key']}_overlay.jpg", num_landmarks)
        rows.append({
            'dataset': dataset_name,
            'image_id': sample.image_id,
            'class_idx': sample.class_idx,
            'orientation': sample.orientation,
            'experiment_key': row['experiment_key'],
            'reconstruction_error': row['reconstruction_error'],
            'valid_landmark_count': int(valid_mask(sample, num_landmarks=num_landmarks).sum()),
            'shape_plot_stem': str(shape_plot_path),
            'overlay_path': '' if overlay_path is None else str(overlay_path),
        })
    report = pd.DataFrame(rows)
    save_table(report, OUTPUT_ROOT / dataset_name.lower() / 'outliers' / f'{dataset_name.lower()}_top_reconstruction_outliers.csv')
    return report


baby_outliers = save_outlier_report(babyland72_samples, baby_reconstruction_errors, 'BabyLand72', BABYLAND72_LANDMARK_COUNT)
infant_outliers = save_outlier_report(infantface_samples, infant_reconstruction_errors, 'InfantFace', INFANTFACE_LANDMARK_COUNT)
baby_outliers.head()


,dataset,image_id,class_idx,orientation,experiment_key,reconstruction_error,valid_landmark_count,shape_plot_stem,overlay_path
0,BabyLand72,face_bcn_16,3,quarter_right,global_68_procrustes,3.631971e-30,58,/Users/jocareher/Library/CloudStorage/OneDrive...,/Users/jocareher/Library/CloudStorage/OneDrive...
1,BabyLand72,face_bcn_19,3,quarter_right,global_68_procrustes,3.483510e-30,65,/Users/jocareher/Library/CloudStorage/OneDrive...,/Users/jocareher/Library/CloudStorage/OneDrive...
2,BabyLand72,face_bcn_202,4,right,global_68_procrustes,3.395303e-30,46,/Users/jocareher/Library/CloudStorage/OneDrive...,/Users/jocareher/Library/CloudStorage/OneDrive...
3,BabyLand72,face_bcn_27,2,frontal,global_68_procrustes,3.369296e-30,58,/Users/jocareher/Library/CloudStorage/OneDrive...,/Users/jocareher/Library/CloudStorage/OneDrive...
4,BabyLand72,face_bcn_108,1,quarter_left,global_68_procrustes,3.367768e-30,65,/Users/jocareher/Library/CloudStorage/OneDrive...,/Users/jocareher/Library/CloudStorage/OneDrive...


## Cross-dataset descriptive comparison

These comparisons use only the first 68 landmarks and remain strictly descriptive. No synthetic prior is involved.


In [83]:
def build_comparison_tables() -> dict[str, pd.DataFrame]:
    orientation_comparison = baby_orientation_table.merge(
        infant_orientation_table,
        on='orientation',
        suffixes=('_babyland72', '_infantface'),
    )
    variability_comparison = baby_variability_68[['landmark_idx', 'anatomical_group', 'anatomical_label', 'spatial_std']].merge(
        infant_variability_68[['landmark_idx', 'spatial_std']],
        on='landmark_idx',
        suffixes=('_babyland72', '_infantface'),
    )
    group_variability = variability_comparison.groupby('anatomical_group', as_index=False)[['spatial_std_babyland72', 'spatial_std_infantface']].mean()
    pca_summary = variance_summary_df[variance_summary_df['scope_name'] == 'global_68'].copy()
    return {
        'orientation_comparison': orientation_comparison,
        'variability_comparison': variability_comparison,
        'group_variability_comparison': group_variability,
        'pca_variance_comparison': pca_summary,
    }


def plot_group_variability_comparison(group_df: pd.DataFrame) -> None:
    fig, ax = plt.subplots(figsize=(10, 5), constrained_layout=True)
    x = np.arange(len(group_df))
    width = 0.36
    ax.bar(x - width / 2, group_df['spatial_std_babyland72'], width=width, color=DATASET_COLORS['BabyLand72'], label='BabyLand72')
    ax.bar(x + width / 2, group_df['spatial_std_infantface'], width=width, color=DATASET_COLORS['InfantFace'], label='InfantFace')
    ax.set_xticks(x)
    ax.set_xticklabels(group_df['anatomical_group'], rotation=35)
    ax.set_ylabel('Mean spatial std (pixels)')
    ax.set_title('Anatomical-group variability comparison (first 68 landmarks)')
    ax.legend(frameon=False)
    save_figure(fig, OUTPUT_ROOT / 'comparisons' / 'figures' / 'anatomical_group_variability_comparison')


comparison_tables = build_comparison_tables()
for name, table in comparison_tables.items():
    save_table(table, OUTPUT_ROOT / 'comparisons' / 'tables' / f'{name}.csv')
plot_group_variability_comparison(comparison_tables['group_variability_comparison'])
comparison_tables['group_variability_comparison']


,anatomical_group,spatial_std_babyland72,spatial_std_infantface
0,face_contour,0.186453,442.169676
1,inner_lip,0.195258,423.328295
2,left_eye,0.134577,432.089416
3,left_eyebrow,0.161394,440.496364
4,nose_base,0.160910,415.405295
5,nose_bridge,0.169148,414.693492
6,outer_lip,0.200734,424.470458
7,right_eye,0.140932,405.722186
8,right_eyebrow,0.163846,413.256484


## Final summary

This section assembles a compact summary table and leaves an interpretation template for notes.


In [84]:
def build_final_summary() -> pd.DataFrame:
    rows = [
        {
            'dataset': 'BabyLand72',
            'num_samples': len(babyland72_samples),
            'mean_valid_landmark_count': float(babyland72_sample_summary['valid_landmark_count'].mean()) if not babyland72_sample_summary.empty else np.nan,
            'most_variable_landmark': baby_variability_68.sort_values('spatial_std', ascending=False).iloc[0]['anatomical_label'] if not baby_variability_68.empty else '',
            'most_variable_group': correlation_summary_baby.iloc[0]['anatomical_group'] if not correlation_summary_baby.empty else '',
        },
        {
            'dataset': 'InfantFace',
            'num_samples': len(infantface_samples),
            'mean_valid_landmark_count': float(infantface_sample_summary['valid_landmark_count'].mean()) if not infantface_sample_summary.empty else np.nan,
            'most_variable_landmark': infant_variability_68.sort_values('spatial_std', ascending=False).iloc[0]['anatomical_label'] if not infant_variability_68.empty else '',
            'most_variable_group': correlation_summary_infant.iloc[0]['anatomical_group'] if not correlation_summary_infant.empty else '',
        },
    ]
    return pd.DataFrame(rows)


final_summary_df = build_final_summary()
save_table(final_summary_df, OUTPUT_ROOT / 'tables' / 'final_dataset_summary.csv')
final_summary_df


,dataset,num_samples,mean_valid_landmark_count,most_variable_landmark,most_variable_group
0,BabyLand72,311,48.093248,face_contour_9,outer_lip
1,InfantFace,405,68.000000,face_contour_17,face_contour


### Interpretation checklist

Use this section to summarize the main findings after running the notebook.

- Dataset sizes and orientation balance:
- BabyLand72 visibility constraints and chosen PCA subset:
- Most variable landmarks in BabyLand72:
- Most variable landmarks in InfantFace:
- Most variable anatomical groups in BabyLand72:
- Most variable anatomical groups in InfantFace:
- Does global PCA appear dominated by yaw/orientation?
- Do class-conditioned PCAs look more interpretable?
- Do score distributions look approximately Gaussian or clearly multimodal?
- Which samples are the strongest reconstruction outliers?
- What are the key geometric differences between BabyLand72 and InfantFace?
